## Supplementary Figure 9, 10, 11

Enviorment: env 1

In [1]:
# env 1
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib as mpl

import sys
sys.path.append("/gpfs/data/shenhavlab/users/ca3261/Visionary-AI-HDP/code/utils/")
from utils import bl_clean, wrap_labels

In [2]:
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
plt.rcParams["font.family"] = "DejaVu Sans"
dpi = 300

In [3]:
bl_name = {
    "graph_feats_cum_size_dist_feats__Random_Forest": "graph_feats[cum_size_dist_feats]_Random_Forest",
    "graph_feats_mixed_feat_set__Logistic_Regression": "graph_feats[mixed_feat_set]_Logistic_Regression", 
    "graph_feats_graph_cols__Random_Forest": "graph_feats[graph_cols]_Random_Forest",
    "graph_feats_tort_cols__Logistic_Regression": "graph_feats[tort_cols]_Logistic_Regression",
    "graph_feats_topo_lengths_feats__Logistic_Regression": "graph_feats[topo_lengths_feats]_Logistic_Regression",
    "graph_feats_geom_feats__Logistic_Regression": "graph_feats[geom_feats]_Logistic_Regression",
    "processed_dan_tree_feats_angle_feats_dan__Random_Forest": "processed_dan_tree_feats[angle_feats_dan]_Random_Forest",
    "graph_feats_mixed_feat_set__Random_Forest": "graph_feats[mixed_feat_set]_Random_Forest",
    "graph_feats_graph_cols__Logistic_Regression": "graph_feats[graph_cols]_Logistic_Regression",
    "processed_dan_tree_feats_angle_feats_dan__XGBoost": "processed_dan_tree_feats[angle_feats_dan]_XGBoost",
    "graph_feats_nest_num_feats__XGBoost": "graph_feats[nest_num_feats]_XGBoost",
    "processed_dan_tree_feats_angle_feats_dan__Logistic_Regression": "processed_dan_tree_feats[angle_feats_dan]_Logistic_Regression"
}

In [4]:
all_feats = {
    "graph_feats[mixed_feat_set]": ['Branches','Distance Between Branches','Square Curvature Tortuosity','Loops','FDS','Avg Topological Length Superior','Avg Topological Length Inferior'],
    "box_counting_df": ["PC " + str(i + 1) for i in range(10)],
    "tree_feats_len_real_bottom": ["PC " + str(i + 1) for i in range(10)],
    "graph_feats[cum_size_dist_feats]": ['Avg Cummulative Loop Area','Sum Cummulative Loop Area','1st Quartile Cummulative Loop Area','2nd Quartile Cummulative Loop Area','3rd Quartile Cummulative Loop Area','4th Quartile Cummulative Loop Area'],
    "graph_feats[graph_cols]": ['Nodes','Components','Edge Nodes','Branches','Avg Direct Distance'],
    "graph_feats[tort_cols]": ['Sinuosity','Distance Inflection Count','Density','Linear Regression','Square Curvature','Absolute Curvature'],
    'graph_feats[topo_lengths_feats]': ['avg_topo_length_top','avg_topo_length_bottom','median_topo_length_top','median_topo_length_bottom',
                             'avg_topo_length_weight_top','avg_topo_length_weight_bottom','median_topo_length_weight_top','median_topo_length_weight_bottom'],
    'graph_feats[geom_feats]': ['vein_density_len','mean_vein_distances','mean_areole_area','areole_density'],
    'processed_dan_tree_feats[angle_feats_dan]': ['Avg Angle','Median Angle','Sum Angle'],
    'old_tree_feats_diameter': ["PC " + str(i + 1) for i in range(10)],
    'tree_feats_len_bottom': ["PC " + str(i + 1) for i in range(10)],
    "old_tree_feats_assymetry": ["PC " + str(i + 1) for i in range(10)],
    "old_tree_feats_len": ["PC " + str(i + 1) for i in range(10)],
    "graph_feats[nest_num_feats]": ['nest_num_unw','nest_num_w','avg_nest_ratios','avg_nest_ratios_w','sum_nest_ratios','sum_nest_ratios_w'],
    "tda_outward": ["PC " + str(i + 1) for i in range(10)],
}

In [5]:
def remove_model(bl):
    if "XGBoost" in bl:
        return bl.split("_XGBoost")[0]
    if "Logistic_Regression" in bl:
        return bl.split("_Logistic_Regression")[0]
    if "Random_Forest" in bl:
        return bl.split("_Random_Forest")[0]

In [6]:
def get_pw_feat_imp_comps(data_name, run, pw):
    all_unique_feats = []
    for feats in all_feats.values():
        for f in feats:
            if f not in all_unique_feats:
                all_unique_feats.append(f)
    
    palette = sns.color_palette("tab20", n_colors=len(all_unique_feats))
    feat_color_map = {feat: palette[i] for i, feat in enumerate(all_unique_feats)}
    
    # create a darker version of each color for edges/lines
    def darken_color(color, factor=0.6):
        """Darken an RGB(A) color by multiplying RGB channels by factor."""
        return tuple([c * factor for c in color[:3]] + [1.0])
    
    feat_edge_color_map = {feat: darken_color(color) for feat, color in feat_color_map.items()}
    
    with open(f"{data_path}/{data_name}{pw}/{run}_stable/final_fi_no_repeats_bls_0.5.pkl", "rb") as f:
        bls = pickle.load(f)
   
    for bl in bls:
        if not ("mixed" in bl or "cum" in bl or "cols" in bl):
            continue
        print(bl)
        if "__" in bl:
            original_bl = bl
            bl = bl_name[bl]
        with open(f"{data_path}/{data_name}{pw}/{run}_stable/rev_strat_pec_hc_pw{pw}_pickles_temp/hp_tuning_{bl}.pkl", "rb") as f:
            hp = list(pickle.load(f)[0].keys())[0]
        with open(f"{data_path}/{data_name}{pw}/{run}/rev_strat_pec_hc_pw{pw}_pickles_temp/import/importance_{bl}.pickle", "rb") as f:
            importances = pickle.load(f)[hp]
        feat_imps = []
        print(f"there are {len(list(importances.keys()))} ids")
        for idx in range(len(list(importances.keys()))):
            id = list(importances.keys())[idx]
            if "Logistic" in bl:
                imp = importances[id][0]
                feats = all_feats[remove_model(bl)]
                imp = imp / sum(np.abs(imp))
            else:
                feats = all_feats[remove_model(bl)]
                imp = importances[id]
                imp = imp / sum(imp)
            feat_imps.append(dict(zip(feats, imp)))
        print(f"there are {len(list(importances.keys()))} feature imps")
        feat_imps_df = pd.DataFrame(feat_imps).fillna(0)
        print(f"there are {len(feat_imps_df.columns)} columns")
        ordered_cols = list(feat_imps_df.mean().sort_values(ascending=False).index)
        
        fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True, dpi=dpi)
        
        box_data = [feat_imps_df[col].values for col in ordered_cols]
        bp = ax.boxplot(
            box_data,
            positions=range(len(ordered_cols)),
            widths=0.6,
            patch_artist=True,  # Required to fill boxes with color
            flierprops=dict(marker='D', markersize=4),
        )
        
        for i, col in enumerate(ordered_cols):
            fill_color = feat_color_map[col]
            edge_color = feat_edge_color_map[col]
            
            bp['boxes'][i].set_facecolor(fill_color)
            bp['boxes'][i].set_edgecolor(edge_color)
            
            bp['whiskers'][i * 2].set_color(edge_color)
            bp['whiskers'][i * 2 + 1].set_color(edge_color)
            
            bp['caps'][i * 2].set_color(edge_color)
            bp['caps'][i * 2 + 1].set_color(edge_color)
            
            bp['medians'][i].set_color(edge_color)
            
            bp['fliers'][i].set_markeredgecolor(edge_color)
            bp['fliers'][i].set_markerfacecolor(edge_color)
        
        ax.set_xticks(range(len(ordered_cols)))
        ax.set_xticklabels(ordered_cols, rotation=90)
        
        bl = bl_clean[original_bl]
        plt.title(f"PW{pw} {bl} LOO Feature Importance")
        plt.savefig(f'/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/src/analysis/final_figures/final_submission/supplementary/fig_9_10_11/pw_{pw}_{bl.lower()}.pdf', bbox_inches='tight')
        plt.show()

In [7]:
mpl.use("Agg")

In [8]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/"
run = "run_20260407"
data_name = "rev_strat_pec_hc_pw"

In [9]:
for i in range(1, 6):
    get_pw_feat_imp_comps(data_name, run, i)

graph_feats_cum_size_dist_feats__Random_Forest
there are 278 ids
there are 278 feature imps
there are 6 columns
graph_feats_mixed_feat_set__Logistic_Regression
there are 278 ids
there are 278 feature imps
there are 7 columns
graph_feats_graph_cols__Random_Forest
there are 278 ids
there are 278 feature imps
there are 5 columns
graph_feats_tort_cols__Logistic_Regression
there are 278 ids
there are 278 feature imps
there are 6 columns
graph_feats_cum_size_dist_feats__Random_Forest
there are 278 ids
there are 278 feature imps
there are 6 columns
graph_feats_graph_cols__Random_Forest
there are 278 ids
there are 278 feature imps
there are 5 columns
graph_feats_mixed_feat_set__Random_Forest
there are 278 ids
there are 278 feature imps
there are 7 columns
graph_feats_graph_cols__Logistic_Regression
there are 275 ids
there are 275 feature imps
there are 5 columns
graph_feats_tort_cols__Logistic_Regression
there are 275 ids
there are 275 feature imps
there are 6 columns
graph_feats_mixed_feat_se

In [10]:
def get_test_feat_imp(data_name, run, feat_set_to_check):
    """
    loo heatmap
    """
    
    features = all_feats[feat_set_to_check]
    
    model_types_to_check = ['Logistic_Regression', 'Random_Forest', 'XGBoost']
    model_display = {'Logistic_Regression': 'LR', 'Random_Forest': 'RF', 'XGBoost': 'XGB'}
    
    pw_data = {}
    
    for train_pw in range(1, n_pop + 1):
        found_models = []
        
        for model_type_name in model_types_to_check:
            bl_name_full = f"{feat_set_to_check}_{model_type_name}"
            path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/{data_name}{train_pw}/{run}/{data_name}{train_pw}_final_bls/trained_model_{bl_name_full}.pkl"
            
            try:
                with open(path, "rb") as f:
                    data = pickle.load(f)
                
                # extract model from pipeline
                pipe = data["pipe"]
                model = pipe.named_steps[model_type_name]
                
                # get importances
                if hasattr(model, 'feature_importances_'):
                    importances = model.feature_importances_
                elif hasattr(model, 'coef_'):
                    importances = np.abs(model.coef_[0])
                    importances = importances / importances.sum()
                else:
                    continue
                
                found_models.append((model_type_name, importances))
                
            except (FileNotFoundError, OSError):
                continue
        
        if len(found_models) > 1:
            print(f"PW{train_pw} has MULTIPLE models for {feat_set_to_check}: {[model_display[m[0]] for m in found_models]}")
            model_type_name, importances = found_models[0]
            pw_data[train_pw] = (f"{model_display[model_type_name]}*", importances)
        elif len(found_models) == 1:
            model_type_name, importances = found_models[0]
            pw_data[train_pw] = (model_display[model_type_name], importances)
        else:
            print(f"PW{train_pw}: No models found for {feat_set_to_check}, skipping.")
    
    if not pw_data:
        print(f"No models found for {feat_set_to_check} in any PW. Nothing to plot.")
        return
    
    # build heatmap only with PWs that have models
    pw_cols = [f"PW{pw}" for pw in sorted(pw_data.keys())]
    heatmap_df = pd.DataFrame(0.0, index=features, columns=pw_cols)
    
    for train_pw, (display_name, importances) in pw_data.items():
        for feat, imp in zip(features, importances):
            heatmap_df.loc[feat, f"PW{train_pw}"] = imp
    
    # sort by prevalence (number of non-zero PWs), then by mean importance as tiebreaker
    heatmap_df['prevalence'] = (heatmap_df[pw_cols] > 0).sum(axis=1)
    heatmap_df['mean_imp'] = heatmap_df[pw_cols].mean(axis=1)
    heatmap_df = heatmap_df.sort_values(['prevalence', 'mean_imp'], ascending=[False, False])
    heatmap_df = heatmap_df.drop(columns=['prevalence', 'mean_imp'])
    
    # build column labels with model info (only for PWs with models)
    col_labels = []
    for pw in sorted(pw_data.keys()):
        display_name = pw_data[pw][0]
        col_labels.append(f"PW{pw}\n({display_name})")
    
    # get display name from bl_clean dictionary
    feat_set_display = None
    for key, value in bl_clean.items():
        if key.startswith(feat_set_to_check.replace(']', '_').replace('[', '_')):
            display = value
            for suffix in [' LR', ' RF', ' XGB']:
                if display.endswith(suffix):
                    display = display[:-len(suffix)]
                    break
            feat_set_display = display
            break
    
    if feat_set_display is None:
        feat_set_display = feat_set_to_check.replace('_', ' ').replace('[', ' ').replace(']', '')
    
    # plot heatmap
    n_pw_plotted = len(pw_data)
    fig, ax = plt.subplots(figsize=(n_pw_plotted * 2 + 2, len(heatmap_df) * 0.4 + 2), dpi=dpi)
    
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".3f",
        cmap="rocket_r",
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Base Learner Feature Importance'},
        vmin=0,
        mask=(heatmap_df == 0)
    )
    
    ax.set_xticklabels(col_labels)
    ax.set_yticklabels(wrap_labels(heatmap_df.index, 19))
    ax.set_xlabel("Control Group (Model)")
    ax.set_ylabel("Feature")
    ax.set_title(f"")
    
    plt.tight_layout()
    plt.savefig(f'/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/src/analysis/final_figures/final_submission/supplementary/fig_9_10_11/fi_{feat_set_to_check}.pdf')
    plt.show()

In [11]:
data_name = "rev_strat_pec_hc_pw"
train_test_run_models = "run_20260416_model_stable_fi_no_repeats"
n_pop = 5
model_type = "XGBoost"
get_test_feat_imp(data_name, train_test_run_models, "graph_feats[mixed_feat_set]")

PW5: No models found for graph_feats[mixed_feat_set], skipping.


In [12]:
get_test_feat_imp(data_name, train_test_run_models, "graph_feats[cum_size_dist_feats]")

PW3: No models found for graph_feats[cum_size_dist_feats], skipping.


In [13]:
get_test_feat_imp(data_name, train_test_run_models, "graph_feats[graph_cols]")

PW4: No models found for graph_feats[graph_cols], skipping.
